<a href="https://colab.research.google.com/github/smriti3003/Agents/blob/main/AgentCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU "google-genai>=2.9.0"

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
# @title
!pip install -q llama-index llama-index-llms-google-genai requests

import os
import requests

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool



# -------------------------------------
# Gemini Model
# -------------------------------------

llm = GoogleGenAI(
    model="gemini-3.5-flash",
    api_key=GEMINI_API_KEY
)
# -------------------------------------
# SENSOR 1
# -------------------------------------

def get_weather(city: str) -> str:

    url = f"https://wttr.in/{city}?format=j1"

    response = requests.get(url)

    if response.status_code != 200:
        return "Unable to retrieve weather information."

    data = response.json()

    current_weather = data["current_condition"][0]

    temperature = current_weather["temp_C"]
    humidity = current_weather["humidity"]
    description = current_weather["weatherDesc"][0]["value"]

    return f"""
    City: {city}
    Temperature: {temperature}°C
    Humidity: {humidity}%
    Weather: {description}
    """


# -------------------------------------
# ACTUATOR
# -------------------------------------

def save_recommendation(recommendation: str) -> str:

    with open("weather_recommendation.txt", "w") as file:
        file.write(recommendation)

    return "Recommendation successfully saved."


# -------------------------------------
# CREATE TOOLS
# -------------------------------------

weather_sensor = FunctionTool.from_defaults(
    fn=get_weather
)

save_actuator = FunctionTool.from_defaults(
    fn=save_recommendation
)


# -------------------------------------
# CREATE AGENT
# -------------------------------------

agent = FunctionAgent(
    llm=llm,
    tools=[
        weather_sensor,
        save_actuator
    ],
    system_prompt="""
    You are a weather recommendation agent.

    Use the weather sensor to obtain current weather information.
    Analyze the information.
    Recommend appropriate clothing and whether an umbrella is needed.
    Finally, save your recommendation using the save_recommendation tool.
    """
)




In [ ]:
print(llm)
print(agent)

callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7f4c35f5a900> rate_limiter=None system_prompt=None messages_to_prompt=<function messages_to_prompt at 0x7f4c3cdedee0> completion_to_prompt=<function default_completion_to_prompt at 0x7f4c3faddd00> output_parser=None pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'> query_wrapper_prompt=None model='gemini-3.5-flash' temperature=None context_window=None max_retries=3 is_function_calling_model=True cached_content=None built_in_tool=None file_mode='hybrid'
name='Agent' description='An agent that can perform a task' system_prompt='\n    You are a weather recommendation agent.\n\n    Use the weather sensor to obtain current weather information.\n    Analyze the information.\n    Recommend appropriate clothing and whether an umbrella is needed.\n    Finally, save your recommendation using the save_recommendation tool.\n    ' tools=[<llama_index.core.tools.function_tool.FunctionTool object at 0x7f4c35f5a

In [ ]:
try:
    response = await agent.run(
        "Check the current weather in Hyderabad. "
        "Tell me what I should wear and whether I should carry an umbrella. "
        "Save your recommendation to a file."
    )
    print(response)

except Exception as e:
    import traceback
    traceback.print_exc()

The current weather in Hyderabad is **26°C** with **79% humidity** and **mist**. 

### Recommendations:
* **What to wear:** Wear light, breathable clothing (such as cotton) to stay comfortable in the warm and highly humid conditions. 
* **Umbrella:** Since it is misty and damp, carrying a light umbrella or a water-resistant jacket is recommended as a precaution against the dampness, although heavy rain is not currently indicated.

Your recommendation has been successfully saved!
